# Strong Equivalence Principle (Nordtvedt Effect)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/notebooks/LFM_Nordtvedt_SEP.ipynb)

## What This Notebook Demonstrates

The **Strong Equivalence Principle** (SEP) says all objects fall the same way regardless of mass or internal energy.

In LFM, we test this by dropping wave packets of different amplitudes (= different masses) in a $\chi$-gradient:

- **Test A**: Same shape, different amplitude (9$\times$ mass ratio) — should fall identically
- **Test B** (control): Different width, same amplitude — dispersion $\neq$ SEP violation

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CHI_0 = 19.0
C = 1.0
N = 512
L = 200.0
dx = L / N
dt = 0.04
n_steps = 5000
CHI_SLOPE = 0.02
DAMPING = 0.99995
x = np.arange(N) * dx
chi = CHI_0 + CHI_SLOPE * x

def run_drop(amplitude, width, label):
    pos0 = 60.0
    E = amplitude * np.exp(-((x - pos0) / width)**2)
    E_prev = E.copy()
    trajectory = []
    for step in range(n_steps):
        lap = np.zeros(N)
        lap[1:-1] = (E[:-2] + E[2:] - 2*E[1:-1]) / dx**2
        E_new = 2*E - E_prev + dt**2 * (C**2 * lap - chi**2 * E)
        E_new *= DAMPING
        E_new[0] = E_new[-1] = 0
        E_prev, E = E.copy(), E_new.copy()
        E2 = E**2
        total = np.sum(E2)
        if total > 1e-20:
            trajectory.append(np.sum(x * E2) / total)
        else:
            trajectory.append(trajectory[-1] if trajectory else pos0)
    return np.array(trajectory)

# Test A: Same shape, different amplitude
traj_heavy = run_drop(3.0, 3.0, 'Heavy (A=3)')
traj_light = run_drop(1.0, 3.0, 'Light (A=1)')
# Test B (control): Different width
traj_wide = run_drop(1.0, 6.0, 'Wide (w=6)')

# SEP analysis
n_valid = min(len(traj_heavy), len(traj_light))
diff_A = np.abs(traj_heavy[:n_valid] - traj_light[:n_valid])
travel = np.abs(traj_heavy[:n_valid] - traj_heavy[0])
frac_diff = diff_A / np.maximum(travel, 1e-10)

# Use latter half where motion is significant
half = n_valid // 2
max_frac = np.max(frac_diff[half:])

diff_B = np.abs(traj_heavy[:n_valid] - traj_wide[:n_valid])
frac_diff_B = diff_B / np.maximum(travel, 1e-10)
max_frac_B = np.max(frac_diff_B[half:])

print('=' * 60)
print('NORDTVEDT SEP TEST')
print('=' * 60)
print(f'Test A (same shape, 9x mass ratio):')
print(f'  Max fractional difference: {max_frac:.6f}')
print(f'  SEP holds (< 0.01): {max_frac < 0.01}')
print(f'\nTest B (different width - control):')
print(f'  Max fractional difference: {max_frac_B:.6f}')
print(f'  Diverges (dispersion): {max_frac_B > 0.01}')
print(f'\nH0 REJECTED (SEP confirmed): {max_frac < 0.01}')

In [ ]:
t_arr = np.arange(n_valid) * dt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(t_arr, traj_heavy[:n_valid], 'r-', lw=2, label='Heavy (A=3, w=3)')
ax.plot(t_arr, traj_light[:n_valid], 'b--', lw=2, label='Light (A=1, w=3)')
ax.plot(t_arr, traj_wide[:n_valid], 'g:', lw=2, label='Wide (A=1, w=6)')
ax.set_xlabel('Time')
ax.set_ylabel('Center of mass position')
ax.set_title('Test A: Same shape, different mass')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.semilogy(t_arr[half:], frac_diff[half:], 'b-', lw=2, label='Test A (diff amplitude)')
ax.semilogy(t_arr[half:], frac_diff_B[half:], 'g-', lw=2, label='Test B (diff width)')
ax.axhline(0.01, color='r', ls='--', label='1% threshold')
ax.set_xlabel('Time')
ax.set_ylabel('Fractional trajectory difference')
ax.set_title('SEP Test: Trajectory Divergence')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Nordtvedt SEP: Universality of Free Fall', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Key Result

- Wave packets with **different amplitudes** (masses) follow the **same trajectory** — SEP holds
- Different **widths** diverge due to dispersion, not SEP violation
- LFM naturally satisfies the Strong Equivalence Principle
- The Nordtvedt parameter $\eta$ is effectively zero in LFM